In [1]:
# ==========================================
# FILE 2: BERT MIL with WEIGHTED DICE LOSS
# ==========================================
!pip install -q transformers datasets torch scikit-learn accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter, defaultdict
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    BertPreTrainedModel,
    BertModel,
    Trainer,
    TrainingArguments,
    EvalPrediction
)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# CONFIGURATION & DATA LOADING
# ==========================================
MODEL_ID = "bert-base-uncased"
MAX_LENGTH = 256
CHUNK_STRIDE = 160
MAX_CHUNKS = 16
BATCH_SIZE = 8

print("\n--- Loading Data ---")
dataset = load_dataset("ailsntua/QEvasion")
train_df = dataset['train'].filter(lambda x: x['evasion_label'] is not None and x['evasion_label'] != "")
labels_list = sorted(train_df.unique('evasion_label'))
num_labels = len(labels_list)
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}

def encode_train_labels(batch):
    return {"labels": label2id[batch['evasion_label']]}

train_dataset = train_df.map(encode_train_labels)

# --- Calculating Class Weights (Used by Weighted Dice) ---
print("\n--- Calculating Class Weights ---")
train_labels_list = train_dataset['labels']
label_counts = Counter(train_labels_list)
total_samples = len(train_labels_list)

class_weights = []
for i in range(num_labels):
    count = label_counts.get(i, 0)
    if count == 0: count = 1
    weight = total_samples / (num_labels * count)
    class_weights.append(weight)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# ==========================================
# DATASET & COLLATOR
# ==========================================
class MILChunkingDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len, stride, max_chunks, is_test=False):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.max_chunks = max_chunks
        self.is_test = is_test

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        question = row['question']
        long_answer = row['interview_answer']
        label = 0 if self.is_test else row['labels']

        answer_tokens = self.tokenizer(long_answer, add_special_tokens=False)['input_ids']
        tokens_per_chunk = self.max_len - 64
        windows = [[]] if len(answer_tokens) == 0 else [
            answer_tokens[i : i + tokens_per_chunk] for i in range(0, len(answer_tokens), self.stride)
        ][:self.max_chunks]

        chunk_input_ids, chunk_masks, chunk_types = [], [], []
        for window in windows:
            encoded = self.tokenizer(
                question, self.tokenizer.decode(window),
                padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt'
            )
            chunk_input_ids.append(encoded['input_ids'].squeeze(0))
            chunk_masks.append(encoded['attention_mask'].squeeze(0))
            chunk_types.append(encoded['token_type_ids'].squeeze(0))

        return {
            "input_ids": torch.stack(chunk_input_ids),
            "attention_mask": torch.stack(chunk_masks),
            "token_type_ids": torch.stack(chunk_types),
            "labels": torch.tensor(label, dtype=torch.long)
        }

@dataclass
class MILDataCollator:
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_chunks = max(f["input_ids"].shape[0] for f in features)
        b_ids, b_mask, b_type, b_lbl = [], [], [], []

        for f in features:
            sid, smask, stype = f["input_ids"], f["attention_mask"], f["token_type_ids"]
            curr_chunks, seq_len = sid.shape
            pad_n = max_chunks - curr_chunks

            if pad_n > 0:
                p_ids = torch.zeros((pad_n, seq_len), dtype=torch.long)
                pid = torch.cat([sid, p_ids], 0)
                pmask = torch.cat([smask, p_ids], 0)
                ptype = torch.cat([stype, p_ids], 0)
            else:
                pid, pmask, ptype = sid, smask, stype

            b_ids.append(pid); b_mask.append(pmask); b_type.append(ptype); b_lbl.append(f["labels"])

        return {
            "input_ids": torch.stack(b_ids), "attention_mask": torch.stack(b_mask),
            "token_type_ids": torch.stack(b_type), "labels": torch.stack(b_lbl)
        }

# ==========================================
# LOSS FUNCTION: WEIGHTED DICE LOSS
# ==========================================
class WeightedDiceLoss(nn.Module):
    def __init__(self, weights=None, smooth=1e-6):
        super(WeightedDiceLoss, self).__init__()
        self.weights = weights
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=logits.shape[1]).float()

        intersection = (probs * targets_one_hot).sum(dim=0)
        union = probs.sum(dim=0) + targets_one_hot.sum(dim=0)

        dice_score = (2. * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1 - dice_score

        if self.weights is not None:
            # Ensure weights are on the same device as loss
            if self.weights.device != dice_loss.device:
                self.weights = self.weights.to(dice_loss.device)

            # Apply weights to the loss per class and normalize
            weighted_loss = (dice_loss * self.weights).sum() / self.weights.sum()
            return weighted_loss

        return dice_loss.mean()

# ==========================================
# CUSTOM MODEL (BERT + ATTN MIL + WEIGHTED DICE)
# ==========================================
class BertForMIL(BertPreTrainedModel):
    def __init__(self, config, class_weights=None):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        self.attention_layer = nn.Linear(config.hidden_size, 1)
        self.dropout = nn.Dropout(getattr(config, "classifier_dropout_prob", config.hidden_dropout_prob))
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # --- MODIFIED: Init Weighted Dice Loss ---
        self.class_weights = class_weights
        self.loss_fct = WeightedDiceLoss(weights=self.class_weights)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        batch_size, num_chunks, seq_len = input_ids.shape
        outputs = self.bert(
            input_ids=input_ids.view(-1, seq_len),
            attention_mask=attention_mask.view(-1, seq_len),
            token_type_ids=token_type_ids.view(-1, seq_len) if token_type_ids is not None else None
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        attn_scores = self.attention_layer(cls_output).view(batch_size, num_chunks)
        attn_scores = attn_scores.masked_fill(~torch.any(attention_mask > 0, dim=-1), -65000.0)
        attn_weights = F.softmax(attn_scores, dim=1)

        context_vector = torch.sum(cls_output.view(batch_size, num_chunks, -1) * attn_weights.unsqueeze(-1), dim=1)
        logits = self.classifier(self.dropout(context_vector))

        loss = None
        if labels is not None:
            loss = self.loss_fct(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

# ==========================================
# TRAINER & EXECUTION
# ==========================================
class MultiAnnotatorTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        output = self.predict(eval_dataset, metric_key_prefix="test")
        preds = np.argmax(output.predictions, axis=1)
        pred_labels = [self.model.config.id2label[p] for p in preds]
        hf_test_data = eval_dataset.dataset

        tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
        all_classes = set()

        for i, pred_label in enumerate(pred_labels):
            anns = [hf_test_data[i]['annotator1'], hf_test_data[i]['annotator2'], hf_test_data[i]['annotator3']]
            true_set = set([a for a in anns if a])
            all_classes.add(pred_label); all_classes.update(true_set)
            if pred_label in true_set: tp[pred_label] += 1
            else:
                fp[pred_label] += 1
                for true_cls in true_set: fn[true_cls] += 1

        f1_scores = []
        for cls in all_classes:
            p = tp[cls] / (tp[cls] + fp[cls]) if (tp[cls] + fp[cls]) > 0 else 0.0
            r = tp[cls] / (tp[cls] + fn[cls]) if (tp[cls] + fn[cls]) > 0 else 0.0
            f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
            f1_scores.append(f1)

        macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
        metrics = {f"{metric_key_prefix}_f1_macro": macro_f1}
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
train_ds = MILChunkingDataset(train_dataset, tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS, is_test=False)
test_ds = MILChunkingDataset(dataset['test'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS, is_test=True)

model = BertForMIL.from_pretrained(
    MODEL_ID, num_labels=num_labels, id2label=id2label, label2id=label2id,
    class_weights=class_weights_tensor
)

training_args = TrainingArguments(
    output_dir="./Bert_MIL_Evasion_WeightedDice",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=15,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = MultiAnnotatorTrainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, data_collator=MILDataCollator()
)

print("\n--- Starting Training (Weighted Dice) ---")
trainer.train()

# ==========================================
# PREDICTION GENERATION
# ==========================================
print("\n" + "="*50 + "\nFINAL REPORT (WEIGHTED DICE)\n" + "="*50)
output = trainer.predict(test_ds)
preds = np.argmax(output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]

output_file = "prediction_weighted_dice"
with open(output_file, "w") as f:
    for label in pred_labels:
        f.write(f"{label}\n")
print(f"✅ Saved predictions to '{output_file}'")
!head -n 5 prediction_weighted_dice

Using device: cuda

--- Loading Data ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Some weights of BertForMIL were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['attention_layer.bias', 'attention_layer.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Token indices sequence length is longer than the specified maximum sequence length for this model (866 > 512). Running this sequence through the model will result in indexing errors



--- Starting Training (Weighted Dice) ---


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.913700,No log,0.312118
2,0.873300,No log,0.363406
3,0.830300,No log,0.408638
4,0.840400,No log,0.412129
5,0.825200,No log,0.404302
6,0.786600,No log,0.373136
7,0.766400,No log,0.397131
8,0.769100,No log,0.361182
9,0.722400,No log,0.378305
10,0.745900,No log,0.343628



FINAL REPORT (WEIGHTED DICE)


✅ Saved predictions to 'prediction_weighted_dice'
General
General
Implicit
Implicit
Explicit
